In [ ]:
import csv
from itertools import combinations
import networkx as nx

In [ ]:
DATA_FOLDER = "../data/"
CSV_PREDICATE_ACTOR = f"{DATA_FOLDER}predicate_sentence_actor.csv"
CSV_EDGES_COOCURRENCE_PREDICATE_NETWORK = f"{DATA_FOLDER}edges_sentence_cooccurrence_predicate_network.csv"
GEPHI_ACTOR_ACTOR_NETWORK_FILE = f"{DATA_FOLDER}gephi/actor_actor_sentence_network.gexf"
GEPHI_ACTOR_STANCE_NETWORK_FILE = f"{DATA_FOLDER}gephi/actor_predicate_type_sentence_network.gexf"
LIST_OF_YEARS = ["1996", "1997", "1998", "1999", "2000", "2001", "2002", "2003", "2005", "2006", "2007", "2024"]
YEAR_FROM_1996_TO_1999 = ["1996", "1997", "1998", "1999"]
YEAR_FROM_2000_TO_2007 = ["2000", "2001", "2002", "2003", "2005", "2006", "2007"]
YEAR_FROM_2024_TO_2024 = ["2024"]

In [ ]:
def get_actor_with_predicate_cooccurrences_by_sentence_for_year(
    year: str,
) -> dict[tuple[str, str, str], dict[str, int]]:
    sentences = {}
    with open(CSV_PREDICATE_ACTOR, mode="r", encoding="utf-8") as file:
        csvFile = csv.DictReader(file)

        for row in csvFile:
            if row["year"].strip() != year:
                continue

            sentence_id = row["sentence_id"].strip()
            actor = row["actor"].strip()
            predicate_type = row["predicate_type"].strip().lower()

            if not actor or predicate_type not in {"support", "opposition"}:
                continue

            if sentence_id not in sentences:
                sentences[sentence_id] = []

            sentences[sentence_id].append((actor, predicate_type))

    cooccurrences = {}

    for sentence_id, rows in sentences.items():
        actors = sorted({actor for actor, _ in rows})

        if len(actors) < 2:
            continue

        actor_predicates = {}
        for actor, predicate_type in rows:
            if actor not in actor_predicates:
                actor_predicates[actor] = []
            actor_predicates[actor].append(predicate_type)

        for actor1, actor2 in combinations(actors, 2):
            key = (year, actor1, actor2)

            if key not in cooccurrences:
                cooccurrences[key] = {
                    "Positive_count": 0,
                    "Negative_count": 0,
                }

            for p1 in actor_predicates[actor1]:
                for p2 in actor_predicates[actor2]:
                    if p1 == p2:
                        cooccurrences[key]["Positive_count"] += 1
                    else:
                        cooccurrences[key]["Negative_count"] += 1

    return cooccurrences

In [ ]:
def build_csv_edges_coocurrence_network_with_predicate(cooccurrences: dict[tuple[str, str, str], dict[str, int]]):
    with open(CSV_EDGES_COOCURRENCE_PREDICATE_NETWORK, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        for (year, source, target), counts in sorted(cooccurrences.items()):
            weight = counts["Positive_count"] - counts["Negative_count"]
            writer.writerow([
                source,
                target,
                weight,
                year,
                year,
                counts["Positive_count"],
                counts["Negative_count"],
                "positive" if weight > 0 else "negative" if weight < 0 else "neutral"
            ])

In [ ]:
with open(CSV_EDGES_COOCURRENCE_PREDICATE_NETWORK, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(['Source', 'Target', 'Weight', 'Start', 'End', 'Positive_count', 'Negative_count', 'Sign'])

for year in LIST_OF_YEARS:
    print(f"Processing year {year}...")
    foo = get_actor_with_predicate_cooccurrences_by_sentence_for_year(year)
    build_csv_edges_coocurrence_network_with_predicate(foo)
    print(f"Finished processing year {year}.\n")

# Graph actor-actor

In [ ]:
A = nx.Graph()

with open(CSV_EDGES_COOCURRENCE_PREDICATE_NETWORK, mode="r", encoding="utf-8") as file:
        csvFile = csv.DictReader(file)
        for row in csvFile:
            A.add_edge(
                row["Source"],
                row["Target"],
                weight=abs(int(row["Weight"])),
                start=row["Start"],
                end=row["End"],
                positive_count=row["Positive_count"],
                negative_count=row["Negative_count"],
                sign=row["Sign"]
            )

nx.write_gexf(A, GEPHI_ACTOR_ACTOR_NETWORK_FILE, encoding="utf-8")

print("Nodes:", A.number_of_nodes())
print("Edges:", A.number_of_edges())
print("Density:", nx.density(A))
print("Connected components:", nx.number_connected_components(A))

In [ ]:
import csv
import networkx as nx
import matplotlib.pyplot as plt

CSV_FILE = CSV_EDGES_COOCURRENCE_PREDICATE_NETWORK

def build_period_graph(csv_file, start_year, end_year):
    G = nx.Graph()

    with open(csv_file, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            start = int(row["Start"])
            end = int(row["End"])

            # ici on garde les arêtes dont l'année tombe dans la période
            if start_year <= start <= end_year:
                G.add_edge(
                    row["Source"],
                    row["Target"],
                    weight=abs(int(row["Weight"])),
                    start=start,
                    end=end,
                    positive_count=int(row["Positive_count"]),
                    negative_count=int(row["Negative_count"]),
                    sign=row["Sign"]
                )
    return G

def get_stats(G):
    pos_edges = sum(1 for _, _, d in G.edges(data=True) if d["sign"] == "positive")
    neg_edges = sum(1 for _, _, d in G.edges(data=True) if d["sign"] == "negative")

    return {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "positive_edges": pos_edges,
        "negative_edges": neg_edges
    }

# Construire les 2 graphes
G_9699 = build_period_graph(CSV_FILE, 1996, 1999)
G_0007 = build_period_graph(CSV_FILE, 2000, 2007)

stats_9699 = get_stats(G_9699)
stats_0007 = get_stats(G_0007)

print("1996–1999:", stats_9699)
print("2000–2007:", stats_0007)

# Figure 4
metrics = ["nodes", "edges", "positive_edges", "negative_edges"]
labels = ["Nodes", "Edges", "Positive", "Negative"]

values_9699 = [stats_9699[m] for m in metrics]
values_0007 = [stats_0007[m] for m in metrics]

x = range(len(metrics))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar([i - width/2 for i in x], values_9699, width=width, label="1996–1999")
plt.bar([i + width/2 for i in x], values_0007, width=width, label="2000–2007")

plt.xticks(list(x), labels)
plt.title("Comparison of actor–actor networks by period")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import csv
import networkx as nx
import matplotlib.pyplot as plt

CSV_FILE = CSV_EDGES_COOCURRENCE_PREDICATE_NETWORK

def build_period_graph(csv_file, start_year, end_year):
    G = nx.Graph()

    with open(csv_file, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            year = int(row["Start"])

            if start_year <= year <= end_year:
                source = row["Source"]
                target = row["Target"]
                weight = abs(int(row["Weight"]))

                if G.has_edge(source, target):
                    G[source][target]["weight"] += weight
                else:
                    G.add_edge(source, target, weight=weight)

    return G

def top_weighted_degree(G, top_n=5):
    wd = dict(G.degree(weight="weight"))
    top = sorted(wd.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return top

G_9699 = build_period_graph(CSV_FILE, 1996, 1999)
G_0007 = build_period_graph(CSV_FILE, 2000, 2007)

top_9699 = top_weighted_degree(G_9699, top_n=5)
top_0007 = top_weighted_degree(G_0007, top_n=5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1996–1999
actors_9699 = [x[0] for x in top_9699]
values_9699 = [x[1] for x in top_9699]
axes[0].barh(actors_9699, values_9699)
axes[0].set_title("Top 5 actors by weighted degree\n1996–1999")
axes[0].invert_yaxis()

# 2000–2007
actors_0007 = [x[0] for x in top_0007]
values_0007 = [x[1] for x in top_0007]
axes[1].barh(actors_0007, values_0007)
axes[1].set_title("Top 5 actors by weighted degree\n2000–2007")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Graph biparti actor-predicate_type

In [ ]:
B = nx.Graph()
sentence_counts: dict[int, int] = {}

# list with different actors for each sentence_id
with open(CSV_PREDICATE_ACTOR, mode="r", encoding="utf-8") as file:
    csvFile = csv.DictReader(file)
    for row in csvFile:
        if row["predicate_type"] == "other": continue

        sentence_id = int(row["sentence_id"])
        sentence_counts[sentence_id] = sentence_counts.get(sentence_id, 0) + 1

with open(CSV_PREDICATE_ACTOR, mode="r", encoding="utf-8") as file:
    csvFile = csv.DictReader(file)
    for row in csvFile:
        if row["predicate_type"] == "other": continue

        sentence_id = int(row["sentence_id"])
        if sentence_counts.get(sentence_id, 0) < 2: continue

        actor = row["actor"].strip()
        negotiation_point = row["negotiation_point"].strip()

        B.add_node(actor, bipartite="actor")
        B.add_node(negotiation_point, bipartite="negotiation_point")

        B.add_edge(
            actor,
            negotiation_point,
            year=int(row["year"]),
            sentence_id=sentence_id,
            predicate=row["predicate"],
            predicate_type=row["predicate_type"],
        )

nodes_to_remove = [
    node
    for node, data in B.nodes(data=True)
    if data.get("bipartite") == "negotiation_point" and B.degree(node) < 2
]

B.remove_nodes_from(nodes_to_remove)

nx.write_gexf(B, GEPHI_ACTOR_STANCE_NETWORK_FILE, encoding="utf-8")

print("Nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())
print("Density:", nx.density(B))
print("Connected components:", nx.number_connected_components(B))